In [ ]:
# Cell 1: 终极环境配置与 SparkSession 初始化
import os
import sys
import subprocess

# 1. 确保安装并导入 graphframes Python 客户端库
try:
    import graphframes
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "graphframes", "-q"])
    import graphframes

# 2. Windows 平台特异性兼容配置 (Anti-Gravity 底层加固)
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = os.environ.get('PATH', '') + r';C:\hadoop\bin'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. 动态挂载 GraphFrames 依赖
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages graphframes:graphframes:0.8.3-spark3.5-s_2.12 pyspark-shell'

from pyspark.sql import SparkSession

# 4. 初始化具备强隔离与防溢出保护的 SparkSession
spark = SparkSession.builder \
    .appName("OfflineGraphRiskFeatureMining") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true") \
    .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 5. 设置 GraphFrames 连通图算法强制要求的 Checkpoint 目录 (Windows 绝对路径)
checkpoint_dir = os.path.abspath("./checkpoints_graph")
os.makedirs(checkpoint_dir, exist_ok=True)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

print(f"✅ SparkSession 启动成功! Spark Version: {spark.version}")
print(f"✅ CheckpointDir 已设置为: {checkpoint_dir}")

In [ ]:
# Cell 2: 模拟 DataFrame 数据准备 (构造包含羊毛党特征的“用户-设备-IP表”和“拉新邀请关系表”)
# 采用纯 JVM Spark SQL 声明式构造，100% 避开 Windows 下 Python Worker 序列化通信缺陷

# 1. 用户-设备-IP 行为日志 (包含黑产设备聚集、IP聚集与正常对照组)
user_device_ip_df = spark.sql("""
    SELECT 'u101' AS user_id, 'dev_gang_01' AS device_id, '10.0.0.1' AS ip UNION ALL
    SELECT 'u102', 'dev_gang_01', '10.0.0.1' UNION ALL
    SELECT 'u103', 'dev_gang_01', '10.0.0.2' UNION ALL
    SELECT 'u104', 'dev_gang_02', '10.0.0.2' UNION ALL
    SELECT 'u105', 'dev_gang_02', '10.0.0.2' UNION ALL
    SELECT 'u106', 'dev_gang_02', '10.0.0.3' UNION ALL
    SELECT 'u201', 'dev_farm_01', '192.168.1.10' UNION ALL
    SELECT 'u202', 'dev_farm_01', '192.168.1.10' UNION ALL
    SELECT 'u203', 'dev_farm_02', '192.168.1.10' UNION ALL
    SELECT 'u204', 'dev_farm_02', '192.168.1.11' UNION ALL
    SELECT 'u301', 'dev_norm_01', '172.16.0.1' UNION ALL
    SELECT 'u302', 'dev_norm_02', '172.16.0.2' UNION ALL
    SELECT 'u303', 'dev_norm_03', '172.16.0.3'
""")

# 2. 裂变拉新邀请关系 (构建稠密团伙子图与树状扩散)
invitation_df = spark.sql("""
    SELECT 'u101' AS inviter_id, 'u102' AS invitee_id UNION ALL
    SELECT 'u101', 'u103' UNION ALL
    SELECT 'u102', 'u104' UNION ALL
    SELECT 'u103', 'u105' UNION ALL
    SELECT 'u104', 'u106' UNION ALL
    SELECT 'u201', 'u202' UNION ALL
    SELECT 'u202', 'u203' UNION ALL
    SELECT 'u203', 'u204' UNION ALL
    SELECT 'u204', 'u201' UNION ALL
    SELECT 'u301', 'u302'
""")

user_device_ip_df.createOrReplaceTempView("user_device_ip")
invitation_df.createOrReplaceTempView("invitation")

print("📋 用户-设备-IP 表数据样例:")
user_device_ip_df.show(5)
print("📋 拉新邀请关系表数据样例:")
invitation_df.show(5)

In [ ]:
# Cell 3: Spark SQL 基础特征提取 (按设备统计共用用户数、IP数)
sql_features_df = spark.sql("""
    WITH device_stats AS (
        SELECT 
            device_id,
            COUNT(DISTINCT user_id) AS device_user_count,
            COUNT(DISTINCT ip) AS device_ip_count
        FROM user_device_ip
        GROUP BY device_id
    )
    SELECT 
        u.user_id,
        u.device_id,
        u.ip,
        d.device_user_count,
        d.device_ip_count
    FROM user_device_ip u
    LEFT JOIN device_stats d ON u.device_id = d.device_id
""")

sql_features_df.createOrReplaceTempView("sql_features")
print("📊 Spark SQL 基础统计特征表:")
sql_features_df.show(truncate=False)

In [ ]:
# Cell 4: GraphFrames 图特征计算 (Connected Components 提取 network_size, PageRank 提取 pagerank)
from graphframes import GraphFrame
from pyspark.sql.functions import count, col

# 1. 构造顶点表 (Vertices) 与边表 (Edges)
# 顶点 ID 必须命名为 "id"，边必须包含 "src" 和 "dst"
vertices_df = spark.sql("""
    SELECT DISTINCT user_id AS id FROM (
        SELECT user_id FROM user_device_ip
        UNION ALL
        SELECT inviter_id AS user_id FROM invitation
        UNION ALL
        SELECT invitee_id AS user_id FROM invitation
    )
""")

edges_df = invitation_df.selectExpr("inviter_id AS src", "invitee_id AS dst")

# 2. 创建 GraphFrame 实例
g = GraphFrame(vertices_df, edges_df)

# 3. 运行连通图算法 (Connected Components) 识别独立团伙并统计团伙网络规模
cc_result = g.connectedComponents()
cluster_sizes = cc_result.groupBy("component").agg(count("id").alias("network_size"))
cc_features = cc_result.join(cluster_sizes, on="component", how="left") \
    .select(col("id").alias("user_id"), col("component"), col("network_size"))

# 4. 运行 PageRank 挖掘网络中的核心头目/关键跳转节点
pr_result = g.pageRank(resetProbability=0.15, maxIter=10)
pr_features = pr_result.vertices.select(col("id").alias("user_id"), col("pagerank"))

# 5. 组合图算法特征
graph_features_df = cc_features.join(pr_features, on="user_id", how="inner")

print("🌐 GraphFrames 图算法拓扑特征:")
graph_features_df.show(truncate=False)

In [ ]:
# Cell 5: 特征融合与持久化 (计算 history_risk_score 并以 overwrite 写入 Parquet)
from pyspark.sql.functions import col, round, least, greatest, lit

# 1. 融合 SQL 统计特征与图网络特征
fused_features_df = sql_features_df.join(
    graph_features_df,
    on="user_id",
    how="left"
).fillna({
    "component": -1,
    "network_size": 1,
    "pagerank": 0.15
})

# 2. 计算综合历史风险分 (history_risk_score, 范围 0.0 ~ 100.0 分)
# 评分规则：
#   - (device_user_count - 1) * 20.0 (设备共用黑产信号)
#   - (device_ip_count - 1) * 10.0 (设备漂移 IP 聚集)
#   - (network_size - 1) * 12.0 (团伙聚集与链条裂变)
#   - pagerank * 25.0 (核心中继节点加权)
final_features_df = fused_features_df.withColumn(
    "history_risk_score",
    round(
        least(
            lit(100.0),
            greatest(
                lit(0.0),
                (col("device_user_count") - 1) * 20.0 +
                (col("device_ip_count") - 1) * 10.0 +
                (col("network_size") - 1) * 12.0 +
                col("pagerank") * 25.0
            )
        ), 2
    )
)

print("🎯 最终离线风控特征大宽表 (Top 10):")
final_features_df.show(10, truncate=False)

# 3. 持久化存储至特征库 (Parquet 格式)
output_path = "../data/features/offline_risk_features.parquet"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

final_features_df.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"💾 特征库已成功持久化至: {output_path}")